In [1]:
import numpy as np
import torch
from torch.utils.data import DataLoader
import os
from autoenkoop import *

DEVICE = "cpu"

In [2]:
class KoopmanDataset(torch.utils.data.Dataset):
    def __init__(self, data, window_size=20):
        # data: (Total_T, 2, W, H)
        self.data = torch.FloatTensor(data).permute(0,3,1,2)
        self.window_size = window_size

    def __len__(self):
        return len(self.data) - self.window_size

    def __getitem__(self, idx):
        # Returns a sequence of length 'window_size'
        return self.data[idx : idx + self.window_size]

In [3]:
datafile = np.load(os.path.join(".", "data", "T1492_x1151_y1_z127_c2.npz"))
data = datafile['timeseries']
dataset = KoopmanDataset(data, window_size=16)
loader = DataLoader(dataset, batch_size=8, shuffle=True)

In [4]:
model = ConvAutoencoderKoop(128)

In [10]:
for X_T in loader:
    B, T, C, H, W = X_T.shape
    X = X_T.view(B * T, C, H, W)
    
    Z, X_recon = model(X)
    
    Z_T = Z.view(B, T, -1)
    Z_T_null = Z_T[:, :-1, :]
    Z_T_shift = Z_T[:, 1:, :]
    Z_null = Z_T_null.reshape(B*(T-1), -1)
    Z_shift = Z_T_shift.reshape(B*(T-1), -1)
    K = torch.linalg.lstsq(Z_null, Z_shift).solution
    
    print(Z_T.shape)
    Z_0 = Z_T[:, 0, :]
    print(Z_0.shape)
    predictions = [Z_0]
    for m in range(1, T):
        # z_m = z_0 @ (K^m)
        Z_m = torch.matmul(predictions[-1], K)
        predictions.append(Z_m)
    Z_pred = torch.stack(predictions, dim=1).view(B*T, -1)
    print(Z_pred.shape)
    X_pred = model.decoder(Z_pred)
    print(X_pred.shape)
    print(X.shape)
    print(X_recon.shape)
    break
    #print(Z_null.shape, Z_shift.shape, X_recon.shape, K.shape)

torch.Size([8, 16, 128])
torch.Size([8, 128])
torch.Size([128, 128])
torch.Size([128, 2, 1151, 127])
torch.Size([128, 2, 1151, 127])
torch.Size([128, 2, 1151, 127])


In [ ]:
dim = 20

matrix = np.random.random((dim, dim))

In [ ]:
eigvalues, eigvectors = np.linalg.eig(matrix)

In [ ]:
eigvalues

In [ ]:
eigvalues

In [ ]:
eigvalues_torch = torch.tensor(eigvalues, device=DEVICE)

In [ ]:
difference_matrix = torch.exp(-torch.abs(eigvalues_torch.unsqueeze(0) - eigvalues_torch.unsqueeze(1))**2)

In [ ]:
difference_matrix

In [ ]:
torch.triu(difference_matrix)

In [ ]:
triuindex = torch.triu_indices(20, 20, offset=1)

In [ ]:
triuindex.shape